# Spark Connect OIDC security test

The shared Spark Connect server registers no catalogs and holds no Polaris credentials: each session
declares its own catalogs, authenticated with the user's OIDC token. This notebook checks that:

1. a catalog authenticated with the user's `OIDC_TOKEN` can read a table;
2. a catalog that is not declared in the session does not exist on the server;
3. a catalog with an invalid token, or with no token at all, is rejected;
4. a catalog the user has no grants on (`aws_forbidden`) is rejected, even with a valid token.

Each negative case is an assertion, so running the notebook fails on any regression.

## Requirements

`users-acl.csv` (in `$AFON_HOME/config`) must authorise the test user for `AFON_CATALOG` (default `aws`),
but not for `AFON_FORBIDDEN_CATALOG` (default `aws_forbidden`). Other users may be authorised for any
catalog, as each user has their own Polaris principal role. For example, testing as `will`:

```
mauro,all
phil,all
will,aws:aws_trino:azure:gcp
```

Catalogs are Polaris catalog names, colon-separated. After changing the ACL, apply it to Polaris for all
catalogs:

```
./afon.sh -a configure-users aws all
```

`AFON_CATALOG` must also contain the table `AFON_TABLE` (default `namespaceph.test_table`, created at setup).

## Running

In a JupyterHub pod, `OIDC_TOKEN` is already injected and the defaults below apply. To run it from your
machine against a deployed stack, use `scripts/smoke/notebook.sh <cloud> oidc`, which gets a token for
`USER_ID`, port-forwards the Spark Connect driver and runs this notebook with the
`jupyter-execute` command.

Settings, all optional apart from `OIDC_TOKEN`: `SPARK_REMOTE`, `POLARIS_DOMAIN_NAME`, `OIDC_DOMAIN_NAME`,
`AFON_CATALOG`, `AFON_FORBIDDEN_CATALOG`, `AFON_TABLE` (as `<namespace>.<table>`).

In [ ]:
import os

from pyspark.sql import SparkSession

OIDC_TOKEN          = os.environ["OIDC_TOKEN"]
SPARK_REMOTE        = os.environ.get("SPARK_REMOTE", "sc://spark-connector-0-driver-svc:15002")
POLARIS_DOMAIN_NAME = os.environ.get("POLARIS_DOMAIN_NAME", "emryspolaris.click")
OIDC_DOMAIN_NAME    = os.environ.get("OIDC_DOMAIN_NAME", "emrysoauth.click")
CATALOG             = os.environ.get("AFON_CATALOG", "aws")
FORBIDDEN_CATALOG   = os.environ.get("AFON_FORBIDDEN_CATALOG", "aws_forbidden")
TABLE               = os.environ.get("AFON_TABLE", "namespaceph.test_table")

SPARK_CATALOG = "test_catalog"


def catalog_configs(catalog_name, token):
    """The session configs for a catalog, authenticated with token (none if token is None)."""
    prefix = f"spark.sql.catalog.{SPARK_CATALOG}"
    configs = {
        prefix: "org.apache.iceberg.spark.SparkCatalog",
        f"{prefix}.type": "rest",
        f"{prefix}.uri": f"https://{POLARIS_DOMAIN_NAME}/api/catalog",
        f"{prefix}.warehouse": catalog_name,
        f"{prefix}.cache-enabled": "false",
        f"{prefix}.header.X-Iceberg-Access-Delegation": "vended-credentials",
        f"{prefix}.oauth2-server-uri": f"https://{OIDC_DOMAIN_NAME}/realms/polaris/protocol/openid-connect/token",
        f"{prefix}.scope": "openid",
    }
    if token is not None:
        configs[f"{prefix}.token"] = token
    return configs


def new_session(configs):
    """A new Spark Connect session, so that no catalog is carried over from another test."""
    builder = SparkSession.builder.appName("SparkConnectOidcTest").remote(SPARK_REMOTE)
    for key, value in configs.items():
        builder = builder.config(key, value)
    return builder.create()


def run(configs, action):
    """Runs action on a new session with configs, and returns its result."""
    spark = new_session(configs)
    try:
        return action(spark)
    finally:
        spark.stop()


def read_table(table):
    return lambda spark: spark.read.table(table).limit(5).collect()


def expect_failure(name, configs, action, markers):
    """Asserts action fails with an error mentioning one of markers."""
    try:
        result = run(configs, action)
    except Exception as e:
        error = str(e)
        assert any(m.lower() in error.lower() for m in markers), \
            f"{name}: failed, but not with any of {markers}:\n{error[:2000]}"
        print(f"PASSED {name}: {error.splitlines()[0][:300]}")
        return
    raise AssertionError(f"{name}: expected a failure, but got {result}")


print(f"Spark Connect: {SPARK_REMOTE}, Polaris: {POLARIS_DOMAIN_NAME}, OIDC: {OIDC_DOMAIN_NAME}")

## 1. The user's token can read a table

In [ ]:
rows = run(catalog_configs(CATALOG, OIDC_TOKEN), read_table(f"{SPARK_CATALOG}.{TABLE}"))
assert rows, f"No rows in {CATALOG}.{TABLE}"
print(f"PASSED read {len(rows)} rows from {CATALOG}.{TABLE} with the user's token")

## 2. The server has no catalogs of its own

In [ ]:
# The catalogs the server used to register for everyone, each under its Polaris name.
for catalog in [CATALOG, FORBIDDEN_CATALOG, "federatedCatalogName"]:
    expect_failure(f"catalog {catalog} not declared in the session", {},
                   lambda spark, catalog=catalog: spark.catalog.setCurrentCatalog(catalog),
                   ["CATALOG_NOT_FOUND"])

## 3. An invalid token, or no token, is rejected

In [ ]:
UNAUTHORIZED = ["NotAuthorized", "Unauthorized", "401", "invalid_grant", "invalid_token", "not authorized"]

expect_failure("invalid token", catalog_configs(CATALOG, "not-a-valid-token"), read_table(f"{SPARK_CATALOG}.{TABLE}"),
               UNAUTHORIZED)
expect_failure("no token", catalog_configs(CATALOG, None), read_table(f"{SPARK_CATALOG}.{TABLE}"),
               UNAUTHORIZED)

## 4. A valid token cannot read a catalog the user has no grants on

In [ ]:
# Listing namespaces needs a privilege on the catalog, whatever it contains - unlike loading a table,
# for which Polaris answers 404 before checking privileges when the table does not exist.
expect_failure(f"no grants on {FORBIDDEN_CATALOG}", catalog_configs(FORBIDDEN_CATALOG, OIDC_TOKEN),
               lambda spark: spark.sql(f"SHOW NAMESPACES IN {SPARK_CATALOG}").collect(),
               ["Forbidden", "403", "NotAuthorized", "not authorized"])